In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


In [ ]:
#Version
from qiskit import *
import qiskit
qiskit.__version__

#Noise and Measurement Error Mitigation
#In this episode of Coding with Qiskit, we learned about noise, and saw how it affects quantum computation on real devices by introducing errors.

#To learn more about measurement error mitigation, check out the section in our open-source Qiskit textbook on this topic.

In [ ]:
#Building a simple quantum circuit
nqubits = 3
circuit = QuantumCircuit(nqubits, nqubits)
circuit.h(0)
circuit.cx(0,1)
circuit.cx(1,2)
circuit.measure([0,1,2], [0,1,2])
circuit.draw(output='mpl')

In [ ]:
#Simulating the quantum circuit
simulator = Aer.get_backend('qasm_simulator')
sim_result = execute(circuit, backend=simulator, shots=1024).result()

from qiskit.visualization import plot_histogram
plot_histogram(sim_result.get_counts(circuit))

In [ ]:
#Running the quantum circuit on a real quantum computer
IBMQ.load_account()
provider = IBMQ.get_provider(hub='ibm-q')
device = provider.get_backend('ibmqx2')
job = execute(circuit, backend=device, shots=1024)
#print(job.job_id())
from qiskit_backend_config import job_monitor
job_monitor(job)

In [ ]:
device_result = job.result()
plot_histogram(device_result.get_counts(circuit))

In [ ]:
#Calibrating measurement errors
#Qiskit Ignis was removed; its two measurement-calibration helpers are small enough to write here.
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

def complete_meas_cal(qr, circlabel='mcal'):
    """One circuit per basis state: prepare |label> and measure (like Ignis' complete_meas_cal)."""
    n = len(qr)
    state_labels = [format(i, f'0{n}b') for i in range(2 ** n)]
    circuits = []
    for label in state_labels:
        qc = QuantumCircuit(n, n, name=f'{circlabel}cal_{label}')
        for q, bit in enumerate(reversed(label)):
            if bit == '1':
                qc.x(q)
        qc.measure(range(n), range(n))
        circuits.append(qc)
    return circuits, state_labels

class MeasurementFilter:
    """Inverts the calibration matrix with a constrained least-squares fit (like Ignis)."""
    def __init__(self, cal_matrix, state_labels):
        self.cal_matrix, self.state_labels = cal_matrix, state_labels

    def _mitigate(self, counts):
        raw = np.array([counts.get(label, 0) for label in self.state_labels], dtype=float)
        shots = raw.sum()
        fun = lambda x: np.sum((raw - self.cal_matrix @ x) ** 2)
        cons = {'type': 'eq', 'fun': lambda x: shots - np.sum(x)}
        x = minimize(fun, raw, method='SLSQP', constraints=cons,
                     bounds=[(0, shots)] * len(raw), tol=1e-6).x
        return {label: v for label, v in zip(self.state_labels, x) if v > 1e-9}

    def apply(self, raw):
        if isinstance(raw, dict):
            return self._mitigate(raw)
        mitigate = self._mitigate
        class _MitigatedResult:
            def get_counts(self, *args):
                return mitigate(raw.get_counts(*args))
        return _MitigatedResult()

class CompleteMeasFitter:
    """Calibration matrix A[i, j] = P(measure state i | prepared state j)."""
    def __init__(self, cal_results, state_labels, circlabel='measurement_calibration'):
        self.state_labels = state_labels
        n = len(state_labels)
        self.cal_matrix = np.zeros((n, n))
        for j, label in enumerate(state_labels):
            counts = cal_results.get_counts(f'{circlabel}cal_{label}')
            shots = sum(counts.values())
            for i, measured in enumerate(state_labels):
                self.cal_matrix[i, j] = counts.get(measured, 0) / shots

    @property
    def filter(self):
        return MeasurementFilter(self.cal_matrix, self.state_labels)

    def plot_calibration(self):
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.matshow(self.cal_matrix, cmap=plt.cm.binary, clim=[0, 1])
        ax.set_xticks(range(len(self.state_labels)), self.state_labels, rotation=90)
        ax.set_yticks(range(len(self.state_labels)), self.state_labels)
        ax.set_xlabel('Prepared State'); ax.set_ylabel('Measured State')
        plt.show()

cal_circuits, state_labels = complete_meas_cal(qr=circuit.qregs[0],
                                               circlabel='measurement_calibration')
print(len(cal_circuits))

#So we have a series of 8 calibration circuits to run -
#we can check what the errors are in each of these circuits
#Why 8 circuits? - So that we can prepare each of the 8 states (000, 001, ..., 111) possible with 3 qubits
cal_circuits[6].draw(output='mpl')


In [ ]:
#Now we create the calibration job on the device
cal_job = execute(cal_circuits,
             backend=device,
             shots=8192,
             optimization_level=0 #so that circuits are not changed from here to the real device
                 )
#print(cal_job.job_id())
job_monitor(cal_job)
cal_results = cal_job.result()

In [ ]:
#Let us plot the histogram of one of the circuits, and 
#see whether we are getting the state with errors or not
plot_histogram(cal_results.get_counts(cal_circuits[2]))

In [ ]:
#Now that we have an error profile we can create a fitter that will account for these errors
#Creating the Measurement Fitter Object
meas_fitter = CompleteMeasFitter(cal_results, state_labels)
#This plot tells us that most of the results are correct but there are few errors (faint grey cells) as well
#Although these are small, they will matter in the computation if not addressed
meas_fitter.plot_calibration()

In [ ]:
#Now we will create a filter - to filter out the errors
meas_filter = meas_fitter.filter
#We now apply the filter on the device results - to create the Mitigated measurement
mitigated_result = meas_filter.apply(device_result)
device_counts = device_result.get_counts(circuit) #original counts ran on the device
mitigated_counts = mitigated_result.get_counts(circuit) #mitigated counts
plot_histogram([device_counts, mitigated_counts], 
               legend=['device, noisy', 'device, mitigated'])

In [ ]:
#We can reuse the calbration circuits and filters on another circuit using same # of qubits
#Running error mitigation on a second circuit
circuit2 = QuantumCircuit(3,3)
circuit2.y(1)
circuit2.h(0)
circuit2.cx(0,1)
circuit2.cx(1,2)
circuit2.measure([0,1,2], [0,1,2])
circuit2.draw(output='mpl')

In [ ]:
#How the circuit should behave in an ideal QC
plot_histogram(
    execute(circuit2, backend=simulator, shots=1024).result().get_counts(circuit2))

In [ ]:
#We run the circuit in the real IBMQ device and plot the results
device_counts_2 = execute(circuit2, backend=device, shots=1024).result().get_counts(circuit2)
plot_histogram(device_counts_2)

In [ ]:
#We apply the previous filter based on the calibration circuits on these device counts
#And check how the results have improved
mitigated_counts_2 = meas_filter.apply(device_counts_2)
plot_histogram([device_counts_2, mitigated_counts_2], legend=['device, noisy','device, mitigated'])

In [ ]:
#If we plot just the mitigated counts, it should look much cleaner
plot_histogram(mitigated_counts_2)